In [151]:
import pandas as pd
import mlflow
import subprocess
import re
import os
from pathlib import Path
import time
from datetime import datetime
import sys


# GetPythonPath

In [152]:
print(sys.executable)
print("Current Time:", datetime.now().strftime("%H:%M:%S"))

c:\Users\peter\anaconda3\python.exe
Current Time: 00:06:39


# Save

In [153]:

########################################################
# Setup
########################################################

#ProjectFolder
os.chdir(r"C:\GitTestOnC\Test4_Oct2\Git")

runName = "Test06_Oct2_12.06___LgDta_LgPrm"
notebookName = "main.ipynb"

mult = 3
add = 5

In [154]:
########################################################
# Open Data
########################################################

df = pd.read_csv("data.csv", header=None)

########################################################
# Calc
########################################################

df_after = add + df * mult
sumResult = df_after.to_numpy().sum()

In [155]:
df

,0,1,2
0,5,6,7
1,8,9,10
2,11,12,13
3,14,15,16


In [156]:
df_after

,0,1,2
0,20,23,26
1,29,32,35
2,38,41,44
3,47,50,53


In [157]:
sumResult

np.int64(438)

In [158]:
########################################################
# Setup MlFlow
########################################################

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(Path(notebookName).stem)

########################################################
# Save Data Version With DVC
########################################################

subprocess.run(["py", "-m", "dvc", "add", "data.csv"], check=True)
subprocess.run(["py", "-m", "dvc", "push"], check=True)

dvcText = Path("data.csv.dvc").read_text()
dvcHash = re.search(r"md5:\s*([0-9a-f]+)", dvcText).group(1)

########################################################
# Save Code + DVC Pointer To Git
########################################################

subprocess.run(["git", "add", notebookName, "data.csv.dvc"], check=True)

hasChanges = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode != 0

if hasChanges:
    subprocess.run(["git", "commit", "-m", f"Code for {runName}"], check=True)
    subprocess.run(["git", "push"], check=True)

########################################################
# Get Git Commit
########################################################

gitCommit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()

########################################################
# Save To MlFlow
########################################################

with mlflow.start_run(run_name=runName) as run:
    runId = run.info.run_id
    mlflow.log_params({"mult": mult, "add": add})
    mlflow.log_metric("sum", sumResult)
    mlflow.set_tag("git_commit", gitCommit)
    mlflow.set_tag("dvc_data_hash", dvcHash)
    mlflow.log_artifact("data.csv.dvc", artifact_path="data")

########################################################
# Print
########################################################

print("Git commit =", gitCommit)
print("DVC hash =", dvcHash)
print("MlFlow experiment =", Path(notebookName).stem)
print("MlFlow run =", runName)
print("MlFlow run ID =", runId)
#PrintTime
print("Current Time:", datetime.now().strftime("%H:%M:%S"))


🏃 View run Test06_Oct2_12.06___LgDta_LgPrm at: http://127.0.0.1:5000/#/experiments/2/runs/d695480acb3b4ce885dcd42b3ee78413
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Git commit = 696f6261f623f490e03e12acea4577a1fbd0637c
DVC hash = a097dcd00e61aa1f67113dfe43f37603
MlFlow experiment = main
MlFlow run = Test06_Oct2_12.06___LgDta_LgPrm
MlFlow run ID = d695480acb3b4ce885dcd42b3ee78413
Current Time: 00:06:44


In [159]:
########################################################
# Export Notebook To HTML
########################################################

Path("results").mkdir(exist_ok=True)
htmlFile = Path(f"results/{runName}.html")

#DeleteOldHtml
if htmlFile.exists(): htmlFile.unlink()

#WaitForVSCodeAutoSave
time.sleep(3)

#ExportHtml
subprocess.run([r"C:\Users\peter\anaconda3\envs\torchgpu\python.exe", "-m", "jupyter", "nbconvert", "--to", "html", "--output", f"{runName}.html", "--output-dir", "results", notebookName], check=True)

#WaitForHtml
while not htmlFile.exists(): time.sleep(0.5)

#WaitForWrite
time.sleep(2)

########################################################
# Save HTML To Same MlFlow Run
########################################################

with mlflow.start_run(run_id=runId):
    mlflow.log_artifact(str(htmlFile), artifact_path="notebook")

########################################################
# Save HTML To GitHub
########################################################

subprocess.run(["git", "add", str(htmlFile)], check=True)

hasResultChanges = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode != 0

if hasResultChanges:
    subprocess.run(["git", "commit", "-m", f"Results for {runName}"], check=True)
    subprocess.run(["git", "push"], check=True)

########################################################
# Tag Experiment
########################################################

existingTags = subprocess.check_output(["git", "tag"], text=True).splitlines()

if runName not in existingTags:
    subprocess.run(["git", "tag", "-a", runName, "-m", runName], check=True)
    subprocess.run(["git", "push", "origin", runName], check=True)




🏃 View run Test06_Oct2_12.06___LgDta_LgPrm at: http://127.0.0.1:5000/#/experiments/2/runs/d695480acb3b4ce885dcd42b3ee78413
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
